# ⚖️ Hybrid Search: Vector Similarity + Metadata Filtering

**The Problem:**
FAISS (and raw mathematical vector distances) are terrible at exact math and strict boundaries. If a user asks: *"Show me running shoes under $50"*, FAISS might return a highly relevant $150 running shoe because the semantic vector of "$150" and "$50" are similar contexts! Pure AI loses the strict e-commerce filter constraints.

**The Solution (Hybrid/Filtered Search):**
In production, Vector Databases like Pinecone or Weaviate handle this natively. They allow you to apply "Pre-Filters" (Metadata limits) *before* calculating Vector nearest neighbors.

In this notebook, we simulate how Hybrid Search works using our local FAISS index alongside pure Pandas DataFrame filtering.

In [ ]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import time

### 1. Load Data (Adding Pricing & Inventory Fields)

In [ ]:
# In the actual app, you would have saved pricing inside your DataFrame in script 01.
# Let's mock a price and inventory column onto our dataset to demonstrate the concept.
df = pd.read_csv("shopify_products_prep.csv")

np.random.seed(42)
df["price"] = np.random.uniform(10.0, 200.0, size=len(df)).round(2)  # Mock price
df["in_stock"] = np.random.choice([True, False], size=len(df), p=[0.8, 0.2]) # Mock inventory 80% instock

print("Loaded Dataset with Hybrid Metadata:")
df[["title", "price", "in_stock"]].head()

In [ ]:
# Reload the Text-Embeddings model and rebuild quick index
model = SentenceTransformer('all-MiniLM-L6-v2')

texts = df["embedding_text"].tolist()
embeddings = model.encode(texts)
embeddings = np.array(embeddings).astype("float32")

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)
print(f"FAISS ready with {index.ntotal} docs.")

### 2. Standard Search (Fails constraints)

In [ ]:
query = "black leather jacket"

query_vec = model.encode([query]).astype("float32")
dist, idxs = index.search(query_vec, 5)

print("Standard Vector Results:")
res_standard = df.iloc[idxs[0]]
display(res_standard[["title", "price", "in_stock"]])

# Notice that FAISS might return products that are Out of Stock, or cost >$150.

### 3. Pre-Filtered Hybrid Search

We first filter our potential target audience using standard structured matching (Pandas), and *then* calculate the similarity vectors. (In Pinecone, this is done under the hood atomically using their `{ "price": { "$lt": 50 } }` syntax).

In [ ]:
def hybrid_search(query, max_price=None, must_be_instock=True, k=5):
    # 1. METADATA FILTER PHASE
    mask = pd.Series([True]*len(df))
    
    if must_be_instock:
        mask = mask & (df["in_stock"] == True)
    
    if max_price is not None:
        mask = mask & (df["price"] <= max_price)
        
    valid_indices = df[mask].index.tolist()
    
    if not valid_indices:
        print("No products match your strict metadata filters!")
        return pd.DataFrame()
    
    print(f"After metadata filtering, {len(valid_indices)} out of {len(df)} candidates remain.")
    
    # 2. VECTOR SEARCH PHASE (Re-indexing only the subset)
    # In production, Cloud DBs do this dynamically without having to rebuild the index
    filtered_embeddings = embeddings[valid_indices]
    
    sub_index = faiss.IndexFlatL2(embeddings.shape[1])
    sub_index.add(filtered_embeddings)
    
    query_vec = model.encode([query]).astype("float32")
    dist, idxs = sub_index.search(query_vec, min(k, len(valid_indices)))
    
    # Map local sub_index results back to global DataFrame valid_indices
    global_idxs = [valid_indices[i] for i in idxs[0]]
    results_df = df.iloc[global_idxs].copy()
    results_df["vector_dist"] = dist[0]
    
    return results_df

In [ ]:
print("\n--- User Request: 'black leather jacket under $50' ---\n")

# Note setting the max_price parameter correctly guides the Hybrid Search
best_matches = hybrid_search("black leather jacket", max_price=50.0, must_be_instock=True, k=5)
display(best_matches[["title", "price", "in_stock", "vector_dist"]])